# AIOps Incident Prediction

Goal: score incident risk from service log metrics.

In [ ]:
from pathlib import Path
import pandas as pd

logs = pd.read_csv(Path('../data/service_logs.csv'), parse_dates=['timestamp'])
logs['is_error'] = logs['status_code'] >= 500
logs['is_slow'] = logs['latency_ms'] >= 800
logs

In [ ]:
features = logs.groupby(['service', 'namespace'], as_index=False).agg(
    requests=('status_code', 'count'),
    error_rate=('is_error', 'mean'),
    slow_rate=('is_slow', 'mean'),
    p95_latency_ms=('latency_ms', lambda s: s.quantile(0.95)),
)
features['incident_risk_score'] = (
    features['error_rate'] * 60
    + features['slow_rate'] * 30
    + (features['p95_latency_ms'] / 1000) * 10
)
features.sort_values('incident_risk_score', ascending=False)

## Interview Talking Points

- Incident prediction can start with explainable scoring.
- Use logs, metrics, deployments, and alerts as features.
- Explainability matters because SREs need to trust recommendations.